In [3]:
from pathlib import Path

DATA_DIR = Path("output")

OUTPUT_DIR = Path("model_outputs")

RANDOM_STATE = 42

In [4]:
dataset_path = Path("output/dataset_winsize5h_where.csv")

In [ ]:
import json

import pandas as pd

where_df = pd.read_csv(
    dataset_path,
    dtype={"fold_id": "int64"},
    converters={
        "window": json.loads,
        "label": json.loads,
    },
)

where_df.head()

,fold_id,window,label
0,0,"[{'from_zone_index': 16, 'to_zone_index': 16, ...","{'from_zone_index': 12, 'to_zone_index': 12}"
1,3,"[{'from_zone_index': 9, 'to_zone_index': 9, 'o...","{'from_zone_index': 17, 'to_zone_index': 17}"
2,2,"[{'from_zone_index': 20, 'to_zone_index': 20, ...","{'from_zone_index': 2, 'to_zone_index': 2}"
3,2,"[{'from_zone_index': 9, 'to_zone_index': 20, '...","{'from_zone_index': 22, 'to_zone_index': 22}"
4,1,"[{'from_zone_index': 2, 'to_zone_index': 2, 'o...","{'from_zone_index': 9, 'to_zone_index': 9}"


In [6]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

OUTAGE_TYPE_TO_ID = {
    "Planned": 0,
    "Auto": 1,
}


class WhereOutageDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        window = row["window"]
        label = row["label"]

        outage_type = torch.tensor(
            [OUTAGE_TYPE_TO_ID[event["outage_type"]] for event in window],
            dtype=torch.long,
        )
        from_zone_indices = torch.tensor(
            [event["from_zone_index"] for event in window],
            dtype=torch.long,
        )
        to_zone_indices = torch.tensor(
            [event["to_zone_index"] for event in window],
            dtype=torch.long,
        )
        target = torch.tensor(
            [label["from_zone_index"], label["to_zone_index"]],
            dtype=torch.long,
        )

        return {
            "outage_type": outage_type,
            "from_zone_indices": from_zone_indices,
            "to_zone_indices": to_zone_indices,
            "target": target,
        }


TRAIN_FOLDS = [0, 1, 2, 3, 4]

train_where_dataset = WhereOutageDataset(
    where_df[where_df["fold_id"].isin(TRAIN_FOLDS)]
)

In [8]:
for i, sample_dict in enumerate(train_where_dataset):
    print(sample_dict)
    if i == 10:
        break

{'outage_type': tensor([0]), 'from_zone_indices': tensor([16]), 'to_zone_indices': tensor([16]), 'target': tensor([12, 12])}
{'outage_type': tensor([0]), 'from_zone_indices': tensor([9]), 'to_zone_indices': tensor([9]), 'target': tensor([17, 17])}
{'outage_type': tensor([0, 0]), 'from_zone_indices': tensor([20, 16]), 'to_zone_indices': tensor([20, 16]), 'target': tensor([2, 2])}
{'outage_type': tensor([0, 0, 0, 0, 0, 0, 0]), 'from_zone_indices': tensor([ 9,  2,  2, 11,  2,  2, 20]), 'to_zone_indices': tensor([20,  2,  2, 11,  2,  2, 20]), 'target': tensor([22, 22])}
{'outage_type': tensor([0, 0]), 'from_zone_indices': tensor([ 2, 20]), 'to_zone_indices': tensor([ 2, 20]), 'target': tensor([9, 9])}
{'outage_type': tensor([1, 0, 0]), 'from_zone_indices': tensor([9, 3, 0]), 'to_zone_indices': tensor([9, 3, 0]), 'target': tensor([8, 8])}
{'outage_type': tensor([0, 0]), 'from_zone_indices': tensor([2, 2]), 'to_zone_indices': tensor([2, 2]), 'target': tensor([15, 15])}
{'outage_type': tensor